<a href="https://colab.research.google.com/github/mohanasudhashanmugam/DeepLearning/blob/main/DL_Objdetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To download and unzip the datasets

In [3]:
!kaggle datasets download -d kipshidze/shoplifting-video-dataset


Dataset URL: https://www.kaggle.com/datasets/kipshidze/shoplifting-video-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 726M/726M [00:04<00:00, 160MB/s]



In [6]:
!unzip -q shoplifting-video-dataset.zip -d ./local_colab_storage

Dataset path Extraction



In [11]:
!ls /content/local_colab_storage/normal | wc -l

90


In [12]:
!ls /content/local_colab_storage/shoplifting | wc -l

92


In [3]:
import cv2
import os
import numpy as np
import argparse

In [4]:
# construct the argument parser and parse the arguments
ap = argparse.ArgumentParser()

ap.add_argument("-d", "--dataset",
	default="/content/local_colab_storage/",
	help="path to input dataset")
args_namespace, unknown = ap.parse_known_args()


# Convert to a dictionary
args = vars(args_namespace)


{'dataset': '/content/local_colab_storage/'}


Frame extraction

In [ ]:
def process_video(video_path, max_frames=16, resize_dim=(224, 224)):
    """
    Opens a video, uniformly extracts a fixed number of frames,
    resizes them, and normalizes pixel values.
    """
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Handle empty or corrupted videos
    if total_frames <= 0:
        cap.release()
        return None

    # Calculate uniform intervals to pick frames across the whole video duration
    # This ensures a 5-second video and a 20-second video both yield exactly 'max_frames'
    frame_indices = np.linspace(0, total_frames - 1, max_frames, dtype=int)

    video_frames = []

    for frame_idx in frame_indices:
        # Set the reader to the specific frame index
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        success, frame = cap.read()

        if not success:
            break

        # 1. Convert color from BGR (OpenCV default) to RGB
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 2. Resize the frame (e.g., to 224x224)
        frame_resized = cv2.resize(frame, resize_dim)

        # 3. Normalize pixel values by dividing by 255.0 (converts 0-255 integers to 0.0-1.0 floats)
        frame_normalized = frame_resized / 255.0

        video_frames.append(frame_normalized)

    cap.release()

    # If the video didn't have enough readable frames, pad it or skip it
    if len(video_frames) < max_frames:
        return None

    # Convert list of frames into a single NumPy array
    # Output shape: (16, 224, 224, 3)
    return np.array(video_frames, dtype=np.float32)

# --- EXAMPLE USAGE ON ONE FILE ---
# (Replace with your actual unzipped path from !ls)
sample_path = "./local_colab_storage/Shoplifting/shoplifting_video_1.mp4"

if os.path.exists(sample_path):
    processed_tensor = process_video(sample_path, max_frames=16, resize_dim=(224, 224))
    print("Video Processed Successfully!")
    print(f"Final Tensor Shape: {processed_tensor.shape}") # Expecting (16, 224, 224, 3)
    print(f"Min pixel value: {processed_tensor.min()}, Max pixel value: {processed_tensor.max()}")
else:
    print("Check your file path! The file does not exist at that location.")


90
